# <span style="color:blue">Softmax regression</span>

This code implements softmax regression where we can use all flower categories.

## <span style="color:#0073e6">Reading the dataset</span>

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

# Convert it into a Pandas dataframe
iris_data = pd.DataFrame(data= np.c_[iris['data'], iris['target']],
                     columns= iris['feature_names'] + ['target'])

# <span style="color:blue">Data preparation</span>

In [2]:
from pickle import FALSE
# Divide into target and features
X = iris_data.drop(labels='target', axis=1)
y = iris_data['target']

# one-hot encode the target values
y = y.to_numpy()
y_enc = (np.arange(3) == y[:, None]).astype(float)


In [3]:
# Split into train and test sets, use stratisfied split because data is small
from sklearn.model_selection import StratifiedShuffleSplit

ss = StratifiedShuffleSplit(n_splits=1, test_size=0.3)
ss.get_n_splits(iris_data)
train_index, test_index = ss.split(X, y).__next__()

X_train = X.iloc[train_index]
y_train = y_enc[train_index]
X_test = X.iloc[test_index]
y_test = y_enc[test_index]
class_count = y_test.shape[1]

# <span style="color:blue">GA code</span>

In [4]:
from deap import base
from deap import creator
from deap import tools
import random

In [5]:
# class_count = 3

In [6]:
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_real", random.uniform, -1.0, 1.0)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_real, 5*class_count)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

Note that when we are creating an individual they have 5 genes per output class. This is because we have a bias/intercept plus four weights per class, because we essentially have a linear regression per class with those terms for each (see the mini-lecture on regression for a reminder).

Below we define the softmax function, and then get the prediction by looping through each class until we have perfomed a regression for each class. Here we have assumed that the weights stored in the individuals are in batches of 5 (bias_a,w1_a,w2_a,w3_a,w4_a,bias_b,w1_b,w2_b,w3_b,w4_b,bias_c,w1_c,w2_c,w3_c,w4_c).

Although the mean squared error would work fine for this problem, it is common to use cross_entropy when using softmax, so we implement that too.

In [7]:
import math

def softmax(xs):
    return np.exp(xs) / sum(np.exp(xs))

def prediction(individual, row):
    raw_predictions = np.zeros(class_count)
    for i in range(class_count):
        raw_predictions[i] = individual[5*i] + sum(i * r for i, r in zip(individual[5*i + 1: 5*i + 5], row))

    return(softmax(raw_predictions))

def cross_entropy(y_pred, y_target):
    return -np.sum(np.log(y_pred) * (y_target), axis=1)

In [8]:
softmax(np.array([1,2,3]))

array([0.09003057, 0.24472847, 0.66524096])

In [9]:
def evalAccuracyTrain(individual, X_train, y_train):
    # Gather predictions
    y_pred = np.zeros((X_train.shape[0], class_count))
    index = 0
    for i, row in X_train.iterrows():
        y_pred[index] = prediction(individual, row)
        index += 1

    # Calculate average cross-entropy
    return np.mean(cross_entropy(y_pred, y_train)),

toolbox.register("evaluate", evalAccuracyTrain)

In [10]:
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0.0, sigma=0.3, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

In [11]:
NGEN, CXPB = 50, 0.2
pop = toolbox.population(n=200)

fitnesses = [toolbox.evaluate(ind, X_train, y_train) for ind in pop]
for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

In [15]:
for g in range(NGEN):
    print("-- Generation %i --" % g)
    offspring = toolbox.select(pop, len(pop))
    offspring = list(map(toolbox.clone, offspring))

    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < CXPB:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    for mutant in offspring:
        toolbox.mutate(mutant)
        del mutant.fitness.values

    invalid_inds = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = [toolbox.evaluate(ind, X_train, y_train) for ind in invalid_inds]
    for ind, fit in zip(invalid_inds, fitnesses):
        ind.fitness.values = fit

    best_ind = tools.selBest(pop, 1)[0]
    print("Best individual is %s, %s" % (best_ind, best_ind.fitness.values))
    pop[:] = offspring

-- Generation 0 --
Best individual is [0.9833935534216661, -0.6153141983022742, 2.5749026058755367, -2.4747536835452917, -2.1055927807927897, 1.7318711472868453, -0.12164417159913374, -0.7497722799584459, -0.4465998087103389, -0.2657344270308409, -2.8669577713838197, -1.7207404819156225, -2.3733183204364514, 2.384142386319284, 3.092788238518204], (np.float64(0.12462958150781056),)
-- Generation 1 --
Best individual is [0.9833935534216661, -0.6153141983022742, 2.5749026058755367, -2.4747536835452917, -2.4827839589683887, 1.7318711472868453, -0.12164417159913374, -0.7497722799584459, -0.4465998087103389, -0.2657344270308409, -2.8669577713838197, -1.7207404819156225, -2.3733183204364514, 2.384142386319284, 3.092788238518204], (np.float64(0.12448650572036396),)
-- Generation 2 --
Best individual is [1.963768534622953, -0.39707313381376413, 2.7670682041299473, -3.2576978451704415, -0.14637440263321216, 1.849846170391037, -0.1454023124390744, -0.24622807878403263, -0.3105867467202096, -0.396

In [17]:
y_pred = np.zeros((X_test.shape[0], class_count))
index = 0
for i, row in X_test.iterrows():
    y_pred[index] = prediction(best_ind, row)
    index += 1

y_pred_am = y_pred.argmax(axis=1)
y_test_am = y_test.argmax(axis=1)
accuracy = np.average(y_pred_am == y_test_am)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

Accuracy: 97.78%
